In [2]:
# ============================================================
# CELL 1: Imports + Modell laden
# ============================================================
import os
import shutil
import torch
import pandas as pd
from tqdm import tqdm
from PIL import Image, UnidentifiedImageError
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Gerät: {device}")

processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to(device)
model.eval()
print("Modell geladen: CLIP-ViT-L14")
print(f"Modell liegt auf: {next(model.parameters()).device}")  # sollte 'cuda:0' zeigen, nicht 'cpu'


# ============================================================
# CELL 2: Bereits ermitteltes Prompt-Set (Detailed für beide Klassen)
# ============================================================
categories = ["human", "non-human"]

mixed_prompts = {
    "human":     "This is a close-up photo of a person or group of people posing for "
                 "the camera. Even if there is a beautiful mountain, landscape, or "
                 "scenic view in the background, the person in the foreground is what "
                 "this photo is about, not the scenery behind them.",
    "non-human": "This is a wide landscape or scenic view where no person is posing "
                 "prominently in the foreground. The terrain, vegetation, wildlife, or "
                 "structures fill the frame; any people present are small, distant, or "
                 "incidental background details."
}

category_texts = [mixed_prompts[cat] for cat in categories]
print("Verwendetes Prompt-Set:", mixed_prompts)


# ============================================================
# CELL 3: Klassifikationsfunktion
# ============================================================
def safe_open(path):
    try:
        return Image.open(path).convert("RGB")
    except (UnidentifiedImageError, OSError) as e:
        print(f"  ⚠ Skipped ({os.path.basename(path)}): {e}")
        return None

BATCH_SIZE = 32
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def classify(image_paths, category_texts):
    text_inputs = processor(text=category_texts, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        text_out = model.get_text_features(**text_inputs)
    text_features = text_out if isinstance(text_out, torch.Tensor) else text_out.pooler_output
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    results, confidences, valid_paths = [], [], []

    for i in tqdm(range(0, len(image_paths), BATCH_SIZE), desc="Batches"):
        batch_paths = image_paths[i : i + BATCH_SIZE]
        loaded = [(p, safe_open(p)) for p in batch_paths]
        loaded = [(p, img) for p, img in loaded if img is not None]
        if not loaded:
            continue

        paths_ok, imgs_ok = zip(*loaded)
        image_inputs = processor(images=list(imgs_ok), return_tensors="pt", padding=True).to(device)

        with torch.no_grad():
            image_out = model.get_image_features(**image_inputs)
        image_features = image_out if isinstance(image_out, torch.Tensor) else image_out.pooler_output
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        similarities = image_features @ text_features.T
        batch_preds  = similarities.argmax(dim=1).cpu().numpy()
        batch_conf   = similarities.max(dim=1).values.cpu().float().numpy()

        results.extend([categories[idx] for idx in batch_preds])
        confidences.extend(batch_conf)
        valid_paths.extend(paths_ok)

    return valid_paths, results, confidences


# ============================================================
# CELL 4: FINALE Klassifikation — dein gesamter Datensatz
# ============================================================
full_dataset_folder = r"C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Human_Non_Human\random_sample_images_human_non_human"
output_csv = r"C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Human_Non_Human\random_sample_human_non_human_classification_detailed_prompts_vitl14_CLIP.csv"

full_image_paths = []
for root, _, files in os.walk(full_dataset_folder):
    for fname in sorted(files):
        ext = os.path.splitext(fname)[1].lower()
        if ext in VALID_EXTENSIONS:
            full_image_paths.append(os.path.join(root, fname))

print(f"Bilder im Gesamtdatensatz gefunden: {len(full_image_paths)}")

valid_paths, results, confidences = classify(full_image_paths, category_texts)

df_full_results = pd.DataFrame({
    "file_name": [os.path.basename(p) for p in valid_paths],
    "file_path": valid_paths,
    "pred_label": results,
    "confidence": confidences
})
df_full_results.to_csv(output_csv, index=False)

print(f"\n✅ Klassifikation abgeschlossen.")
print(f"Gespeichert: {output_csv}")
print("\nVerteilung im Gesamtdatensatz:")
print(df_full_results["pred_label"].value_counts())
print(f"\nGesamt klassifiziert: {len(df_full_results)} Bilder")


# ============================================================
# CELL 5: Bilder nach Vorhersage in Ordner sortieren (Kopieren)
# ============================================================
output_base = r"C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Human_Non_Human\random_sample_human_non_human_classification_vitl14_CLIP_detailed_prompts_images_sorted"

sorted_dirs = {cat: os.path.join(output_base, cat) for cat in categories}
for d in sorted_dirs.values():
    os.makedirs(d, exist_ok=True)

for path, pred in tqdm(zip(valid_paths, results), total=len(valid_paths), desc="Kopiere Bilder"):
    dest = os.path.join(sorted_dirs[pred], os.path.basename(path))
    if not os.path.exists(dest):
        shutil.copy(path, dest)

print("\nBilder sortiert:")
for cat in categories:
    print(f"  {cat}: {results.count(cat)} → {sorted_dirs[cat]}")

Gerät: cuda


Loading weights: 100%|█████████████████████████████████████████████████████████████| 590/590 [00:00<00:00, 4110.55it/s]


Modell geladen: CLIP-ViT-L14
Modell liegt auf: cuda:0
Verwendetes Prompt-Set: {'human': 'This is a close-up photo of a person or group of people posing for the camera. Even if there is a beautiful mountain, landscape, or scenic view in the background, the person in the foreground is what this photo is about, not the scenery behind them.', 'non-human': 'This is a wide landscape or scenic view where no person is posing prominently in the foreground. The terrain, vegetation, wildlife, or structures fill the frame; any people present are small, distant, or incidental background details.'}
Bilder im Gesamtdatensatz gefunden: 300


Batches: 100%|█████████████████████████████████████████████████████████████████████████| 10/10 [04:13<00:00, 25.37s/it]



✅ Klassifikation abgeschlossen.
Gespeichert: C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Human_Non_Human\random_sample_human_non_human_classification_detailed_prompts_vitl14_CLIP.csv

Verteilung im Gesamtdatensatz:
pred_label
non-human    176
human        124
Name: count, dtype: int64

Gesamt klassifiziert: 300 Bilder


Kopiere Bilder: 100%|███████████████████████████████████████████████████████████████| 300/300 [00:00<00:00, 503.80it/s]


Bilder sortiert:
  human: 124 → C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Human_Non_Human\random_sample_human_non_human_classification_vitl14_CLIP_detailed_prompts_images_sorted\human
  non-human: 176 → C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Human_Non_Human\random_sample_human_non_human_classification_vitl14_CLIP_detailed_prompts_images_sorted\non-human
